# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [3]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from IPython.display import display

# ==========================================
# 6. CUSTOM MODULAR PLOTTING CLASS
# ==========================================


class PlottingMethods:
    """Handles granular chart generation and returns HTML-wrapped figures

    for flexible embedding in external applications or dashboards.
    """

    @staticmethod
    def wrap_html(fig) -> str:
        """Helper method to export a Plotly figure to an HTML string."""
        if fig is None:
            return "<p>No data available to plot.</p>"
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @classmethod
    def generate_bar_chart(
        cls,
        df: pd.DataFrame,
        x: str,
        y: str,
        title: str = "Bar Chart",
        color: str = None,
    ) -> str:
        """Generates a standard bar chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.bar(df, x=x, y=y, color=color, title=title, template="plotly_white")
        return cls.wrap_html(fig)

    @classmethod
    def generate_pie_chart(
        cls, df: pd.DataFrame, names: str, values: str, title: str = "Pie Chart"
    ) -> str:
        """Generates a pie chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.pie(df, names=names, values=values, title=title, template="plotly_white")
        return cls.wrap_html(fig)

    @classmethod
    def generate_histogram(
        cls, df: pd.DataFrame, x: str, nbins: int = 30, title: str = "Histogram"
    ) -> str:
        """Generates a histogram chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.histogram(df, x=x, nbins=nbins, title=title, template="plotly_white")
        return cls.wrap_html(fig)


# ==========================================
# MAIN DATA INSPECTOR ENGINE CLASS
# ==========================================


class DataInspector:
    """An end-to-end tool for CSV data ingestion, advanced cleaning,

    feature engineering preparation, and high-level statistical visualization.
    """

    def __init__(self):
        """Initializes the engine with an empty DataFrame."""
        self.df = None

    # ------------------------------------------
    # 1. Data Ingestion & Sanitization
    # ------------------------------------------

    def upload_data(self) -> None:
        """Handles local file uploads within a Google Colab environment.

        Automatically triggers file picker dialog.
        """
        try:
            from google.colab import files

            uploaded = files.upload()
            if not uploaded:
                print("⚠️ Upload cancelled or no file selected.")
                return

            file_name = list(uploaded.keys())[0]
            # Custom garbage string conversions matching assignment requirements
            garbage_values = [
              "?",
              "n/a",
              "$n/a",
              "NULL",
              "null",
              "NaN",
              " ",
              ""
            ]

            self.df = pd.read_csv(io.BytesIO(uploaded[file_name]), na_values=garbage_values)
            print(f"✅ Successfully loaded {file_name}!")
            self._auto_type_correction()

        except ImportError:
            print(
                "❌ google.colab module not found. Run this function inside a Google Colab notebook."
            )
        except Exception as e:
            print(f"❌ Error during file ingestion: {e}")

    def load_from_dataframe(self, dataframe: pd.DataFrame) -> None:
        """Helper framework to parse an in-memory DataFrame (e.g., for testing)."""
        if dataframe is None or dataframe.empty:
            print("⚠️ Provided DataFrame is empty.")
            return


        # Convert garbage text strings across the dataframe manually
        df_cleaned = dataframe.replace(garbage_values, np.nan)

        self.df = df_cleaned.copy()
        self._auto_type_correction()

    def _auto_type_correction(self) -> None:
        """Force-converts columns to numeric types if the conversion does

        not result in an entirely null column.
        """
        if self.df is None:
            return

        for col in self.df.columns:
            # If it's already explicitly numeric, skip
            if pd.api.types.is_numeric_dtype(self.df[col]):
                continue

            # Attempt a numeric conversion
            converted = pd.to_numeric(self.df[col], errors="coerce")

            # Keep conversion only if it didn't completely map out to completely null values
            if not converted.isna().all():
                self.df[col] = converted

    # ------------------------------------------
    # 2. Structural Analysis & Cleaning
    # ------------------------------------------

    def display_summary(self) -> None:
        """Displays row/column counts, a breakdown of numerical vs.

        categorical columns, and previews the first 20 rows.
        """
        if self.df is None:
            print("⚠️ No data loaded. Please upload a dataset first.")
            return

        print("-" * 50)
        print("📊 STRUCTURAL DATA SUMMARY")
        print("-" * 50)
        print(f"Total Rows:    {self.df.shape[0]}")
        print(f"Total Columns: {self.df.shape[1]}")

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print(f"\n🔢 Numerical Columns ({len(num_cols)}):\n   {num_cols}")
        print(f"\n🔤 Categorical Columns ({len(cat_cols)}):\n   {cat_cols}")
        print("\n📋 First 20 Rows Preview:")
        display(self.df.head(20))
        print("-" * 50)

    def handle_missing_values(self, column: str, strategy: str, fill_value=None) -> None:
        """Imputes missing values using mean, median, mode, or a constant value."""
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return

        strategy = strategy.lower()
        if strategy == "mean":
            val = self.df[column].mean()
        elif strategy == "median":
            val = self.df[column].median()
        elif strategy == "mode":
            val = self.df[column].mode()[0] if not self.df[column].mode().empty else np.nan
        elif strategy == "constant":
            val = fill_value
        else:
            print(f"⚠️ Strategy '{strategy}' not recognized. Use mean/median/mode/constant.")
            return

        self.df[column] = self.df[column].fillna(val)
        print(f"✨ Imputed missing entries in '{column}' using '{strategy}' strategy ({val}).")

    def remove_duplicates(self) -> None:
        """Prunes exact row matches from the dataset."""
        if self.df is None:
            return
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        final_count = len(self.df)
        print(f"♻️ Removed {initial_count - final_count} identical duplicate rows.")

    def handle_outliers(self, column: str, action: str = "flag") -> pd.Series:
        """IQR-based outlier detection system. Allows flagging outliers or

        automatically deleting rows containing them.
        """
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return None

        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f"⚠️ Cannot calculate outliers on non-numeric column '{column}'.")
            return None

        q1 = self.df[column].quantile(0.25)
        q3 = self.df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)

        if action.lower() == "delete":
            initial_len = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"🗑️ Dropped {initial_len - len(self.df)} rows containing outliers in '{column}'.")
            return None
        else:
            print(f"🔍 Found {outlier_mask.sum()} outliers out of {len(self.df)} rows in '{column}'.")
            return outlier_mask

    def delete_rows(self, row_indices_str: str) -> None:
        """Accepts a comma-separated user input string of indices to prune matching rows."""
        if self.df is None:
            return
        try:
            indices = [int(idx.strip()) for idx in row_indices_str.split(",") if idx.strip()]
            existing_indices = [idx for idx in indices if idx in self.df.index]
            self.df.drop(index=existing_indices, inplace=True)
            print(f"✂️ Successfully pruned rows: {existing_indices}")
        except Exception as e:
            print(f"❌ Error processing input row indices: {e}")

    def delete_columns(self, columns_str: str) -> None:
        """Accepts a comma-separated user input string of names to drop matching columns."""
        if self.df is None:
            return
        columns_to_drop = [col.strip() for col in columns_str.split(",") if col.strip()]
        existing_cols = [col for col in columns_to_drop if col in self.df.columns]
        self.df.drop(columns=existing_cols, inplace=True)
        print(f"✂️ Successfully dropped columns: {existing_cols}")

    # ------------------------------------------
    # 3. Feature Engineering Preparation (Normalization)
    # ------------------------------------------

    def extract_normalized_numeric_data(self, columns: list, strategy: str = "standard") -> pd.DataFrame:
        """Scales numeric columns using 'minmax', 'standard', or 'robust' (IQR) scaling."""
        if self.df is None or not columns:
            return pd.DataFrame()

        valid_cols = [c for c in columns if c in self.df.columns and pd.api.types.is_numeric_dtype(self.df[c])]
        if not valid_cols:
            return pd.DataFrame()

        # Handle NaNs gracefully by doing a quick isolated backfill/forward fill if any remain
        data_to_scale = self.df[valid_cols].bfill().ffill().fillna(0)

        strategy = strategy.lower()
        if strategy == "minmax":
            scaler = MinMaxScaler()
        elif strategy == "robust":
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        scaled_values = scaler.fit_transform(data_to_scale)
        scaled_df = pd.DataFrame(
            scaled_values,
            columns=[f"{col}_{strategy}" for col in valid_cols],
            index=self.df.index,
        )
        return scaled_df

    def extract_normalized_categorical_data(self, columns: list, strategy: str = "onehot") -> pd.DataFrame:
        """Encodes categorical columns using 'onehot', 'ordinal', or 'uniform' configurations."""
        if self.df is None or not columns:
            return pd.DataFrame()

        valid_cols = [c for c in columns if c in self.df.columns]
        if not valid_cols:
            return pd.DataFrame()

        strategy = strategy.lower()
        encoded_dfs = []

        for col in valid_cols:
            # Handle potential residual NaNs safely as a separate string string category
            working_series = self.df[col].astype(str).fillna("Missing")

            if strategy == "onehot":
                onehot_df = pd.get_dummies(working_series, prefix=col, drop_first=False).astype(int)
                encoded_dfs.append(onehot_df)

            elif strategy in ["ordinal", "uniform"]:
                categories = working_series.unique()
                mapping = {cat: i for i, cat in enumerate(categories)}
                encoded_series = working_series.map(mapping)

                if strategy == "uniform":
                    # Custom transformation: Scale ordinal integers cleanly between 0 and 1
                    max_val = encoded_series.max()
                    if max_val > 0:
                        encoded_series = encoded_series / max_val

                encoded_dfs.append(pd.DataFrame({f"{col}_{strategy}": encoded_series}, index=self.df.index))

        return pd.concat(encoded_dfs, axis=1) if encoded_dfs else pd.DataFrame()

    def merge_features(self, numeric_df: pd.DataFrame, categorical_df: pd.DataFrame) -> pd.DataFrame:
        """Combines original dataset numerical items alongside custom generated encoded features."""
        if self.df is None:
            return pd.DataFrame()

        base_numeric = self.df.select_dtypes(include=[np.number]).copy()
        frames = [base_numeric]

        if numeric_df is not None and not numeric_df.empty:
            frames.append(numeric_df)
        if categorical_df is not None and not categorical_df.empty:
            frames.append(categorical_df)

        return pd.concat(frames, axis=1)

    # ------------------------------------------
    # 4. Advanced Interactive Visualization (Plotly)
    # ------------------------------------------

    def plot_univariate_subplots(self, column: str) -> None:
        """Generates an advanced 3-panel dynamic subplot configuration for a target column:

        Panel 1: Horizontal Box/Violin | Panel 2: Index vs Value Scatter | Panel 3: Histogram
        """
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return

        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f"⚠️ Univariate 3-panel subplots require a numeric column. '{column}' is invalid.")
            return

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=(
                "Distribution (Box/Violin)",
                "Index vs Value Scatter",
                "Frequency Histogram",
            ),
        )
        clean_data = self.df[column].dropna()

        # Panel 1: Violin + Box plot
        fig.add_trace(go.Violin(x=clean_data, name=column, box_visible=True, points="all"), row=1, col=1)
        # Panel 2: Index vs Value
        fig.add_trace(
            go.Scatter(x=clean_data.index, y=clean_data, mode="markers", marker=dict(opacity=0.6)),
            row=1,
            col=2,
        )
        # Panel 3: Histogram
        fig.add_trace(go.Histogram(x=clean_data, nbinsx=25), row=1, col=3)

        fig.update_layout(
            title_text=f"📊 Advanced Univariate Deep Dive: {column}",
            template="plotly_white",
            showlegend=False,
            height=450,
        )
        fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """Autodetects feature datatypes to select the appropriate analytical chart structure:

        - Num-Num: Scatter with OLS Trendline.
        - Cat-Num: Box plot with all data points.
        - Cat-Cat: Grouped Bar chart.
        """
        if self.df is None or col_x not in self.df.columns or col_y not in self.df.columns:
            print("⚠️ Check features; specified columns missing.")
            return

        is_x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        is_y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        # 1. Num-Num Connection
        if is_x_num and is_y_num:
            fig = px.scatter(
                self.df,
                x=col_x,
                y=col_y,
                trendline="ols",
                title=f"Scatter Analysis (Num-Num): {col_x} vs {col_y}",
                template="plotly_white",
            )
        # 2. Cat-Num Connection
        elif (not is_x_num and is_y_num) or (is_x_num and not is_y_num):
            cat = col_x if not is_x_num else col_y
            num = col_y if is_y_num else col_x
            fig = px.box(
                self.df,
                x=cat,
                y=num,
                points="all",
                title=f"Box Plot Analysis (Cat-Num): {cat} vs {num}",
                template="plotly_white",
            )
        # 3. Cat-Cat Connection
        else:
            counts = self.df.groupby([col_x, col_y]).size().reset_index(name="Count")
            fig = px.bar(
                counts,
                x=col_x,
                y="Count",
                color=col_y,
                barmode="group",
                title=f"Grouped Bar Analysis (Cat-Cat): {col_x} vs {col_y}",
                template="plotly_white",
            )

        fig.show()

    def plot_categorical_frequency(self, column: str) -> None:
        """Creates bar charts displaying both raw counts and percentage labels."""
        if self.df is None or column not in self.df.columns:
            return

        freq = self.df[column].fillna("Missing").value_counts().reset_index()
        freq.columns = [column, "Count"]
        total = freq["Count"].sum()
        freq["Percentage"] = (freq["Count"] / total * 100).round(2)

        # Build interactive textual representations to attach inside the bars
        freq["Label"] = freq.apply(lambda r: f"{r['Count']} ({r['Percentage']}%)", axis=1)

        fig = px.bar(
            freq,
            x=column,
            y="Count",
            text="Label",
            title=f"Categorical Profile for: {column}",
            template="plotly_white",
        )
        fig.update_traces(textposition="outside")
        fig.show()

    # ------------------------------------------
    # 5. Deep Statistical Insights
    # ------------------------------------------

    def plot_all_associations_heatmap(self) -> None:
        """Visualizes statistical relationships natively across all mixed data types:

        - Num-Num: Pearson's r
        - Cat-Cat: Cramér's V
        - Mixed (Num-Cat): Eta coefficient (via ANOVA framework)
        """
        if self.df is None:
            return

        # Ensure minimal data prep to avoid script breaking on residual NaNs during calculation
        df_local = self.df.copy()
        cols = df_local.columns.tolist()
        n = len(cols)

        # Structure storage for custom relational matrix construction
        matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                col_a = cols[i]
                col_b = cols[j]

                if i == j:
                    matrix[i, j] = 1.0
                    continue

                is_a_num = pd.api.types.is_numeric_dtype(df_local[col_a])
                is_b_num = pd.api.types.is_numeric_dtype(df_local[col_b])

                # Context A: Numeric to Numeric (Pearson Correlation Coefficient)
                if is_a_num and is_b_num:
                    clean = df_local[[col_a, col_b]].dropna()
                    if len(clean) > 1:
                        r = clean[col_a].corr(clean[col_b])
                        matrix[i, j] = np.abs(r) if not np.isnan(r) else 0.0

                # Context B: Categorical to Categorical (Cramér's V)
                elif not is_a_num and not is_b_num:
                    clean = df_local[[col_a, col_b]].dropna()
                    if not clean.empty:
                        contingency_table = pd.crosstab(clean[col_a], clean[col_b])
                        try:
                            chi2, _, _, _ = chi2_contingency(contingency_table)
                            n_obs = contingency_table.sum().sum()
                            min_dim = min(contingency_table.shape) - 1
                            if n_obs > 0 and min_dim > 0:
                                cramers_v = np.sqrt(chi2 / (n_obs * min_dim))
                                matrix[i, j] = cramers_v
                        except:
                            matrix[i, j] = 0.0

                # Context C: Mixed Association (Eta coefficient via One-Way ANOVA)
                else:
                    num_col = col_a if is_a_num else col_b
                    cat_col = col_b if is_a_num else col_a

                    clean = df_local[[num_col, cat_col]].dropna()
                    groups = clean.groupby(cat_col)[num_col].apply(list).tolist()
                    # Filter out empty structures or single elements
                    groups = [g for g in groups if len(g) > 1]

                    if len(groups) > 1:
                        try:
                            f_stat, p_val = f_oneway(*groups)
                            # Approximate an Eta squared correlation from ANOVA structure elements
                            # Eta = sqrt( (F * df1) / (F * df1 + df2) )
                            df1 = len(groups) - 1
                            df2 = len(clean) - len(groups)
                            if (f_stat * df1 + df2) > 0:
                                eta = np.sqrt((f_stat * df1) / (f_stat * df1 + df2))
                                matrix[i, j] = eta if not np.isnan(eta) else 0.0
                        except:
                            matrix[i, j] = 0.0

        # Build dynamic matrix visual
        fig = go.Figure(
            data=go.Heatmap(
                z=matrix,
                x=cols,
                y=cols,
                colorscale="Viridis",
                zmin=0,
                zmax=1,
                hoverongaps=False,
            )
        )
        fig.update_layout(
            title="🌐 Universal Statistical Association Matrix (Pearson / Cramér's V / Eta)",
            template="plotly_white",
            height=600,
            width=700,
        )
        fig.show()


# ==========================================
# 7. REAL-WORLD DEMONSTRATION WORKFLOW
# ==========================================

if __name__ == "__main__":
    print("🚀 Initializing Real-world Test Using Simulated Titanic Dataset...")

    # Let's generate a mock raw dataframe matching the Titanic profile containing typical anomalies
    mock_titanic_data = {
        "PassengerId": [1, 2, 3, 4, 5, 6, 7, 8],
        "Survived": [0, 1, 1, 1, 0, 0, 0, 1],
        "Pclass": [3, 1, 3, 1, 3, 3, 1, 3],
        "Name": ["Thisara", "Nethmi", "jessica", "Sriyani", "Allen", "Pasindu", "Sampath", "Johnson"],
        "Sex": ["male", "female", "female", "female", "male", "male", "male", "$n/a"],  # Garbage string entry
        "Age": [22.0, 38.0, 26.0, 35.0, 35.0, np.nan, 54.0, 160.0],  # Missing value & an extreme outlier (160)
        "Fare": [7.25, 71.28, 7.92, 53.10, 8.05, 8.46, 51.86, np.nan],  # Missing entry
        "Embarked": ["S", "C", "S", "S", "S", "Q", "NULL", "S"],  # Garbage string entry
    }
    raw_df = pd.DataFrame(mock_titanic_data)

    # Instantiate our data processing machinery
    inspector = DataInspector()

    # Step A: Ingestion with built-in garbage cleaning transformations
    print("\n--- STEP A: DATA INGESTION & TYPE CORRECTION ---")
    inspector.load_from_dataframe(raw_df)
    inspector.display_summary()

    # Step B: Advanced Structural Cleaning & Imputation
    print("\n--- STEP B: DATA CLEANING & IMPUTATION ---")
    inspector.handle_missing_values(column="Age", strategy="median")
    inspector.handle_missing_values(column="Fare", strategy="mean")
    inspector.handle_missing_values(column="Sex", strategy="mode")
    inspector.handle_missing_values(column="Embarked", strategy="mode")

    # Handle dataset outliers gracefully using IQR filters
    print("\n--- HANDLING OUTLIERS ---")
    inspector.handle_outliers(column="Age", action="delete")

    # Verification Step
    print("\n--- RE-CHECKING CLEANED STRUCTURE ---")
    inspector.display_summary()

    # Step C: Normalization and Encoding Preparations
    print("\n--- STEP C: FEATURE NORMALIZATION & ENCODING ---")
    scaled_num = inspector.extract_normalized_numeric_data(columns=["Age", "Fare"], strategy="standard")
    encoded_cat = inspector.extract_normalized_categorical_data(columns=["Sex", "Embarked"], strategy="onehot")

    print("📊 Scaled Numeric Feature View Preview:")
    display(scaled_num.head())

    print("\n📊 Encoded Categorical Feature View Preview:")
    display(encoded_cat.head())

    # Build final combined analysis modeling workspace
    unified_ml_df = inspector.merge_features(scaled_num, encoded_cat)
    print("\n💼 Combined Modeling Master DataFrame:")
    display(unified_ml_df.head())

    # Step D: Advanced Multi-type Visualizations
    print("\n--- STEP D: VISUALIZATION DEMONSTRATIONS ---")
    # 1. Univariate Plot
    inspector.plot_univariate_subplots(column="Fare")

    # 2. Variable Relationships
    inspector.plot_relationship(col_x="Pclass", col_y="Fare")  # Cat vs Num
    inspector.plot_relationship(col_x="Sex", col_y="Embarked")  # Cat vs Cat

    # 3. Categorical Frequencies
    inspector.plot_categorical_frequency(column="Embarked")

    # 4. Association Matrix Across Diverse Data Formats
    inspector.plot_all_associations_heatmap()

    # Step E: Modular HTML Export Proof of Concept
    print("\n--- STEP E: MODULAR HTML GENERATION PROOF ---")
    summary_counts = inspector.df["Sex"].value_counts().reset_index()
    summary_counts.columns = ["Sex", "Count"]

    html_snippet = PlottingMethods.generate_bar_chart(
        df=summary_counts, x="Sex", y="Count", title="Modular Component Plot"
    )
    print(f"📦 Successfully exported standalone chart to HTML String snippet. Length: {len(html_snippet)} chars.")

🚀 Initializing Real-world Test Using Simulated Titanic Dataset...

--- STEP A: DATA INGESTION & TYPE CORRECTION ---
--------------------------------------------------
📊 STRUCTURAL DATA SUMMARY
--------------------------------------------------
Total Rows:    8
Total Columns: 8

🔢 Numerical Columns (5):
   ['PassengerId', 'Survived', 'Pclass', 'Age', 'Fare']

🔤 Categorical Columns (3):
   ['Name', 'Sex', 'Embarked']

📋 First 20 Rows Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,Thisara,male,22.0,7.25,S
1,2,1,1,Nethmi,female,38.0,71.28,C
2,3,1,3,jessica,female,26.0,7.92,S
3,4,1,1,Sriyani,female,35.0,53.10,S
4,5,0,3,Allen,male,35.0,8.05,S
5,6,0,3,Pasindu,male,NaN,8.46,Q
6,7,0,1,Sampath,male,54.0,51.86,NaN
7,8,1,3,Johnson,NaN,160.0,NaN,S


--------------------------------------------------

--- STEP B: DATA CLEANING & IMPUTATION ---
✨ Imputed missing entries in 'Age' using 'median' strategy (35.0).
✨ Imputed missing entries in 'Fare' using 'mean' strategy (29.702857142857145).
✨ Imputed missing entries in 'Sex' using 'mode' strategy (male).
✨ Imputed missing entries in 'Embarked' using 'mode' strategy (S).

--- HANDLING OUTLIERS ---
🗑️ Dropped 1 rows containing outliers in 'Age'.

--- RE-CHECKING CLEANED STRUCTURE ---
--------------------------------------------------
📊 STRUCTURAL DATA SUMMARY
--------------------------------------------------
Total Rows:    7
Total Columns: 8

🔢 Numerical Columns (5):
   ['PassengerId', 'Survived', 'Pclass', 'Age', 'Fare']

🔤 Categorical Columns (3):
   ['Name', 'Sex', 'Embarked']

📋 First 20 Rows Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,Thisara,male,22.0,7.25,S
1,2,1,1,Nethmi,female,38.0,71.28,C
2,3,1,3,jessica,female,26.0,7.92,S
3,4,1,1,Sriyani,female,35.0,53.10,S
4,5,0,3,Allen,male,35.0,8.05,S
5,6,0,3,Pasindu,male,35.0,8.46,Q
6,7,0,1,Sampath,male,54.0,51.86,S


--------------------------------------------------

--- STEP C: FEATURE NORMALIZATION & ENCODING ---
📊 Scaled Numeric Feature View Preview:


...


💼 Combined Modeling Master DataFrame:


,PassengerId,Survived,Pclass,Age,Fare,Age_standard,Fare_standard,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,22.0,7.25,-1.381327,-0.869681,0,1,0,0,1
1,2,1,1,38.0,71.28,0.318768,1.610433,1,0,1,0,0
2,3,1,3,26.0,7.92,-0.956303,-0.843729,1,0,0,0,1
3,4,1,1,35.0,53.10,0.000000,0.906256,1,0,0,0,1
4,5,0,3,35.0,8.05,0.000000,-0.838694,0,1,0,0,1



--- STEP D: VISUALIZATION DEMONSTRATIONS ---


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite




--- STEP E: MODULAR HTML GENERATION PROOF ---
📦 Successfully exported standalone chart to HTML String snippet. Length: 8367 chars.


In [4]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2_contingency, f_oneway
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from IPython.display import display

# ==========================================
# 6. CUSTOM MODULAR PLOTTING CLASS
# ==========================================


class PlottingMethods:
    """Handles granular chart generation and returns HTML-wrapped figures

    for flexible embedding in external applications or dashboards.
    """

    @staticmethod
    def wrap_html(fig) -> str:
        """Helper method to export a Plotly figure to an HTML string."""
        if fig is None:
            return "<p>No data available to plot.</p>"
        return fig.to_html(full_html=False, include_plotlyjs="cdn")

    @classmethod
    def generate_bar_chart(
        cls,
        df: pd.DataFrame,
        x: str,
        y: str,
        title: str = "Bar Chart",
        color: str = None,
    ) -> str:
        """Generates a standard bar chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.bar(df, x=x, y=y, color=color, title=title, template="plotly_white")
        return cls.wrap_html(fig)

    @classmethod
    def generate_pie_chart(
        cls, df: pd.DataFrame, names: str, values: str, title: str = "Pie Chart"
    ) -> str:
        """Generates a pie chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.pie(df, names=names, values=values, title=title, template="plotly_white")
        return cls.wrap_html(fig)

    @classmethod
    def generate_histogram(
        cls, df: pd.DataFrame, x: str, nbins: int = 30, title: str = "Histogram"
    ) -> str:
        """Generates a histogram chart and returns its HTML string."""
        if df is None or df.empty:
            return cls.wrap_html(None)
        fig = px.histogram(df, x=x, nbins=nbins, title=title, template="plotly_white")
        return cls.wrap_html(fig)


# ==========================================
# MAIN DATA INSPECTOR ENGINE CLASS
# ==========================================


class DataInspector:
    """An end-to-end tool for CSV data ingestion, advanced cleaning,

    feature engineering preparation, and high-level statistical visualization.
    """

    def __init__(self):
        """Initializes the engine with an empty DataFrame."""
        self.df = None

    # ------------------------------------------
    # 1. Data Ingestion & Sanitization
    # ------------------------------------------

    def upload_data(self) -> None:
        """Handles local file uploads within a Google Colab environment.

        Automatically triggers file picker dialog.
        """
        try:
            from google.colab import files

            uploaded = files.upload()
            if not uploaded:
                print("⚠️ Upload cancelled or no file selected.")
                return

            file_name = list(uploaded.keys())[0]
            # Custom garbage string conversions matching assignment requirements
            garbage_values = [
              "?",
              "n/a",
              "$n/a",
              "NULL",
              "null",
              "NaN",
              " ",
              ""
            ]

            self.df = pd.read_csv(io.BytesIO(uploaded[file_name]), na_values=garbage_values)
            print(f"✅ Successfully loaded {file_name}!")
            self._auto_type_correction()

        except ImportError:
            print(
                "❌ google.colab module not found. Run this function inside a Google Colab notebook."
            )
        except Exception as e:
            print(f"❌ Error during file ingestion: {e}")

    def load_from_dataframe(self, dataframe: pd.DataFrame) -> None:
        """Helper framework to parse an in-memory DataFrame (e.g., for testing)."""
        if dataframe is None or dataframe.empty:
            print("⚠️ Provided DataFrame is empty.")
            return
        garbage_values = [
              "?",
              "n/a",
              "$n/a",
              "NULL",
              "null",
              "NaN",
              " ",
              ""
        ]

        # Convert garbage text strings across the dataframe manually
        df_cleaned = dataframe.replace(garbage_values, np.nan)

        self.df = df_cleaned.copy()
        self._auto_type_correction()

    def _auto_type_correction(self) -> None:
        """
        Automatically converts object columns to numeric
        only when the majority of values are numeric.
        """

        if self.df is None:
            return

        for col in self.df.columns:

            if pd.api.types.is_numeric_dtype(self.df[col]):
                continue

            converted = pd.to_numeric(
                self.df[col],
                errors="coerce"
            )

            success_rate = converted.notna().mean()

            # Convert only if at least 80% of values are numeric
            if success_rate >= 0.80:
                self.df[col] = converted
    # ------------------------------------------
    # 2. Structural Analysis & Cleaning
    # ------------------------------------------

    def display_summary(self) -> None:
        """Displays row/column counts, a breakdown of numerical vs.

        categorical columns, and previews the first 20 rows.
        """
        if self.df is None:
            print("⚠️ No data loaded. Please upload a dataset first.")
            return

        print("-" * 50)
        print("📊 STRUCTURAL DATA SUMMARY")
        print("-" * 50)
        print(f"Total Rows:    {self.df.shape[0]}")
        print(f"Total Columns: {self.df.shape[1]}")

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print(f"\n🔢 Numerical Columns ({len(num_cols)}):\n   {num_cols}")
        print(f"\n🔤 Categorical Columns ({len(cat_cols)}):\n   {cat_cols}")
        print("\n📋 First 20 Rows Preview:")
        display(self.df.head(20))
        print("-" * 50)

    def handle_missing_values(self, column: str, strategy: str, fill_value=None) -> None:
        """Imputes missing values using mean, median, mode, or a constant value."""
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return

        strategy = strategy.lower()
        if strategy == "mean":
            val = self.df[column].mean()
        elif strategy == "median":
            val = self.df[column].median()
        elif strategy == "mode":
            val = self.df[column].mode()[0] if not self.df[column].mode().empty else np.nan
        elif strategy == "constant":
            val = fill_value
        else:
            print(f"⚠️ Strategy '{strategy}' not recognized. Use mean/median/mode/constant.")
            return

        self.df[column] = self.df[column].fillna(val)
        print(f"✨ Imputed missing entries in '{column}' using '{strategy}' strategy ({val}).")

    def remove_duplicates(self) -> None:
        """Prunes exact row matches from the dataset."""
        if self.df is None:
            return
        initial_count = len(self.df)
        self.df.drop_duplicates(inplace=True)
        final_count = len(self.df)
        print(f"♻️ Removed {initial_count - final_count} identical duplicate rows.")

    def handle_outliers(self, column: str, action: str = "flag") -> pd.Series:
        """IQR-based outlier detection system. Allows flagging outliers or

        automatically deleting rows containing them.
        """
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return None

        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f"⚠️ Cannot calculate outliers on non-numeric column '{column}'.")
            return None

        q1 = self.df[column].quantile(0.25)
        q3 = self.df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_mask = (self.df[column] < lower_bound) | (self.df[column] > upper_bound)

        if action.lower() == "delete":
            initial_len = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f"🗑️ Dropped {initial_len - len(self.df)} rows containing outliers in '{column}'.")
            return None
        else:
            print(f"🔍 Found {outlier_mask.sum()} outliers out of {len(self.df)} rows in '{column}'.")
            return outlier_mask

    def delete_rows(self, row_indices_str: str) -> None:
        """Accepts a comma-separated user input string of indices to prune matching rows."""
        if self.df is None:
            return
        try:
            indices = [int(idx.strip()) for idx in row_indices_str.split(",") if idx.strip()]
            existing_indices = [idx for idx in indices if idx in self.df.index]
            self.df.drop(index=existing_indices, inplace=True)
            print(f"✂️ Successfully pruned rows: {existing_indices}")
        except Exception as e:
            print(f"❌ Error processing input row indices: {e}")

    def delete_columns(self, columns_str: str) -> None:
        """Accepts a comma-separated user input string of names to drop matching columns."""
        if self.df is None:
            return
        columns_to_drop = [col.strip() for col in columns_str.split(",") if col.strip()]
        existing_cols = [col for col in columns_to_drop if col in self.df.columns]
        self.df.drop(columns=existing_cols, inplace=True)
        print(f"✂️ Successfully dropped columns: {existing_cols}")

    # ------------------------------------------
    # 3. Feature Engineering Preparation (Normalization)
    # ------------------------------------------

    def extract_normalized_numeric_data(self, columns: list, strategy: str = "standard") -> pd.DataFrame:
        """Scales numeric columns using 'minmax', 'standard', or 'robust' (IQR) scaling."""
        if self.df is None or not columns:
            return pd.DataFrame()

        valid_cols = [c for c in columns if c in self.df.columns and pd.api.types.is_numeric_dtype(self.df[c])]
        if not valid_cols:
            return pd.DataFrame()

        # Handle NaNs gracefully by doing a quick isolated backfill/forward fill if any remain
        data_to_scale = self.df[valid_cols].bfill().ffill().fillna(0)

        strategy = strategy.lower()
        if strategy == "minmax":
            scaler = MinMaxScaler()
        elif strategy == "robust":
            scaler = RobustScaler()
        else:
            scaler = StandardScaler()

        scaled_values = scaler.fit_transform(data_to_scale)
        scaled_df = pd.DataFrame(
            scaled_values,
            columns=[f"{col}_{strategy}" for col in valid_cols],
            index=self.df.index,
        )
        return scaled_df

    def extract_normalized_categorical_data(self, columns: list, strategy: str = "onehot") -> pd.DataFrame:
        """Encodes categorical columns using 'onehot', 'ordinal', or 'uniform' configurations."""
        if self.df is None or not columns:
            return pd.DataFrame()

        valid_cols = [c for c in columns if c in self.df.columns]
        if not valid_cols:
            return pd.DataFrame()

        strategy = strategy.lower()
        encoded_dfs = []

        for col in valid_cols:
            # Handle potential residual NaNs safely as a separate string string category
            working_series = (
              self.df[col]
              .fillna("Missing")
              .astype(str)
            )

            if strategy == "onehot":
                onehot_df = pd.get_dummies(working_series, prefix=col, drop_first=False).astype(int)
                encoded_dfs.append(onehot_df)

            elif strategy in ["ordinal", "uniform"]:
                categories = working_series.unique()
                mapping = {cat: i for i, cat in enumerate(categories)}
                encoded_series = working_series.map(mapping)

                if strategy == "uniform":
                    # Custom transformation: Scale ordinal integers cleanly between 0 and 1
                    max_val = encoded_series.max()
                    if max_val > 0:
                        encoded_series = encoded_series / max_val

                encoded_dfs.append(pd.DataFrame({f"{col}_{strategy}": encoded_series}, index=self.df.index))

        return pd.concat(encoded_dfs, axis=1) if encoded_dfs else pd.DataFrame()

    def merge_features(self, numeric_df: pd.DataFrame, categorical_df: pd.DataFrame) -> pd.DataFrame:
        """Combines original dataset numerical items alongside custom generated encoded features."""
        if self.df is None:
            return pd.DataFrame()

        base_numeric = self.df.select_dtypes(include=[np.number]).copy()
        frames = [base_numeric]

        if numeric_df is not None and not numeric_df.empty:
            frames.append(numeric_df)
        if categorical_df is not None and not categorical_df.empty:
            frames.append(categorical_df)

        return pd.concat(frames, axis=1)

    # ------------------------------------------
    # 4. Advanced Interactive Visualization (Plotly)
    # ------------------------------------------

    def plot_univariate_subplots(self, column: str) -> None:
        """Generates an advanced 3-panel dynamic subplot configuration for a target column:

        Panel 1: Horizontal Box/Violin | Panel 2: Index vs Value Scatter | Panel 3: Histogram
        """
        if self.df is None or column not in self.df.columns:
            print(f"⚠️ Column '{column}' not found.")
            return

        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f"⚠️ Univariate 3-panel subplots require a numeric column. '{column}' is invalid.")
            return

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=(
                "Distribution (Box/Violin)",
                "Index vs Value Scatter",
                "Frequency Histogram",
            ),
        )
        clean_data = self.df[column].dropna()

        # Panel 1: Violin + Box plot
        fig.add_trace(go.Violin(x=clean_data, name=column, box_visible=True, points="all"), row=1, col=1)
        # Panel 2: Index vs Value
        fig.add_trace(
            go.Scatter(x=clean_data.index, y=clean_data, mode="markers", marker=dict(opacity=0.6)),
            row=1,
            col=2,
        )
        # Panel 3: Histogram
        fig.add_trace(go.Histogram(x=clean_data, nbinsx=25), row=1, col=3)

        fig.update_layout(
            title_text=f"📊 Advanced Univariate Deep Dive: {column}",
            template="plotly_white",
            showlegend=False,
            height=450,
        )
        fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """Autodetects feature datatypes to select the appropriate analytical chart structure:

        - Num-Num: Scatter with OLS Trendline.
        - Cat-Num: Box plot with all data points.
        - Cat-Cat: Grouped Bar chart.
        """
        if self.df is None or col_x not in self.df.columns or col_y not in self.df.columns:
            print("⚠️ Check features; specified columns missing.")
            return

        is_x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        is_y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        # 1. Num-Num Connection
        if is_x_num and is_y_num:
            fig = px.scatter(
                self.df,
                x=col_x,
                y=col_y,
                trendline="ols",
                title=f"Scatter Analysis (Num-Num): {col_x} vs {col_y}",
                template="plotly_white",
            )
        # 2. Cat-Num Connection
        elif (not is_x_num and is_y_num) or (is_x_num and not is_y_num):
            cat = col_x if not is_x_num else col_y
            num = col_y if is_y_num else col_x
            fig = px.box(
                self.df,
                x=cat,
                y=num,
                points="all",
                title=f"Box Plot Analysis (Cat-Num): {cat} vs {num}",
                template="plotly_white",
            )
        # 3. Cat-Cat Connection
        else:
            counts = self.df.groupby([col_x, col_y]).size().reset_index(name="Count")
            fig = px.bar(
                counts,
                x=col_x,
                y="Count",
                color=col_y,
                barmode="group",
                title=f"Grouped Bar Analysis (Cat-Cat): {col_x} vs {col_y}",
                template="plotly_white",
            )

        fig.show()

    def plot_categorical_frequency(self, column: str) -> None:
        """Creates bar charts displaying both raw counts and percentage labels."""
        if self.df is None or column not in self.df.columns:
            return

        freq = self.df[column].fillna("Missing").value_counts().reset_index()
        freq.columns = [column, "Count"]
        total = freq["Count"].sum()
        freq["Percentage"] = (freq["Count"] / total * 100).round(2)

        # Build interactive textual representations to attach inside the bars
        freq["Label"] = freq.apply(lambda r: f"{r['Count']} ({r['Percentage']}%)", axis=1)

        fig = px.bar(
            freq,
            x=column,
            y="Count",
            text="Label",
            title=f"Categorical Profile for: {column}",
            template="plotly_white",
        )
        fig.update_traces(textposition="outside")
        fig.show()

    # ------------------------------------------
    # 5. Deep Statistical Insights
    # ------------------------------------------

    def plot_all_associations_heatmap(self) -> None:
        """Visualizes statistical relationships natively across all mixed data types:

        - Num-Num: Pearson's r
        - Cat-Cat: Cramér's V
        - Mixed (Num-Cat): Eta coefficient (via ANOVA framework)
        """
        if self.df is None:
            return

        # Ensure minimal data prep to avoid script breaking on residual NaNs during calculation
        df_local = self.df.copy()
        cols = df_local.columns.tolist()
        n = len(cols)

        # Structure storage for custom relational matrix construction
        matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                col_a = cols[i]
                col_b = cols[j]

                if i == j:
                    matrix[i, j] = 1.0
                    continue

                is_a_num = pd.api.types.is_numeric_dtype(df_local[col_a])
                is_b_num = pd.api.types.is_numeric_dtype(df_local[col_b])

                # Context A: Numeric to Numeric (Pearson Correlation Coefficient)
                if is_a_num and is_b_num:
                    clean = df_local[[col_a, col_b]].dropna()
                    if len(clean) > 1:
                        r = clean[col_a].corr(clean[col_b])
                        matrix[i, j] = np.abs(r) if not np.isnan(r) else 0.0

                # Context B: Categorical to Categorical (Cramér's V)
                elif not is_a_num and not is_b_num:
                    clean = df_local[[col_a, col_b]].dropna()
                    if not clean.empty:
                        contingency_table = pd.crosstab(clean[col_a], clean[col_b])
                        try:
                            chi2, _, _, _ = chi2_contingency(contingency_table)
                            n_obs = contingency_table.sum().sum()
                            min_dim = min(contingency_table.shape) - 1
                            if n_obs > 0 and min_dim > 0:
                                cramers_v = np.sqrt(chi2 / (n_obs * min_dim))
                                matrix[i, j] = cramers_v
                        except:
                            matrix[i, j] = 0.0

                # Context C: Mixed Association (Eta coefficient via One-Way ANOVA)
                else:
                    num_col = col_a if is_a_num else col_b
                    cat_col = col_b if is_a_num else col_a

                    clean = df_local[[num_col, cat_col]].dropna()
                    groups = clean.groupby(cat_col)[num_col].apply(list).tolist()
                    # Filter out empty structures or single elements
                    groups = [g for g in groups if len(g) > 1]

                    if len(groups) > 1:
                        try:
                            f_stat, p_val = f_oneway(*groups)
                            # Approximate an Eta squared correlation from ANOVA structure elements
                            # Eta = sqrt( (F * df1) / (F * df1 + df2) )
                            df1 = len(groups) - 1
                            df2 = len(clean) - len(groups)
                            if (f_stat * df1 + df2) > 0:
                                eta = np.sqrt((f_stat * df1) / (f_stat * df1 + df2))
                                matrix[i, j] = eta if not np.isnan(eta) else 0.0
                        except:
                            matrix[i, j] = 0.0

        # Build dynamic matrix visual
        fig = go.Figure(
            data=go.Heatmap(
                z=matrix,
                x=cols,
                y=cols,
                colorscale="Viridis",
                zmin=0,
                zmax=1,
                hoverongaps=False,
            )
        )
        fig.update_layout(
            title="🌐 Universal Statistical Association Matrix (Pearson / Cramér's V / Eta)",
            template="plotly_white",
            height=600,
            width=700,
        )
        fig.show()

def missing_value_report(self):

    if self.df is None:
        return

    report = pd.DataFrame({
        "Missing Count": self.df.isna().sum(),
        "Missing %":
        (
            self.df.isna().mean() * 100
        ).round(2)
    })

    display(
        report.sort_values(
            "Missing %",
            ascending=False
        )
    )

# ==========================================
# 7. REAL-WORLD DEMONSTRATION WORKFLOW
# ==========================================

if __name__ == "__main__":
    print("🚀 Initializing Real-world Test Using Simulated Titanic Dataset...")

    # Let's generate a mock raw dataframe matching the Titanic profile containing typical anomalies
    mock_titanic_data = {
        "PassengerId": [1, 2, 3, 4, 5, 6, 7, 8],
        "Survived": [0, 1, 1, 1, 0, 0, 0, 1],
        "Pclass": [3, 1, 3, 1, 3, 3, 1, 3],
        "Name": ["Thisara", "Nethmi", "jessica", "Sriyani", "Allen", "Pasindu", "Sampath", "Johnson"],
        "Sex": ["male", "female", "female", "female", "male", "male", "male", "$n/a"],  # Garbage string entry
        "Age": [22.0, 38.0, 26.0, 35.0, 35.0, np.nan, 54.0, 160.0],  # Missing value & an extreme outlier (160)
        "Fare": [7.25, 71.28, 7.92, 53.10, 8.05, 8.46, 51.86, np.nan],  # Missing entry
        "Embarked": ["S", "C", "S", "S", "S", "Q", "NULL", "S"],  # Garbage string entry
    }
    raw_df = pd.DataFrame(mock_titanic_data)

    # Instantiate our data processing machinery
    inspector = DataInspector()

    # Step A: Ingestion with built-in garbage cleaning transformations
    print("\n--- STEP A: DATA INGESTION & TYPE CORRECTION ---")
    inspector.load_from_dataframe(raw_df)
    inspector.display_summary()

    # Step B: Advanced Structural Cleaning & Imputation
    print("\n--- STEP B: DATA CLEANING & IMPUTATION ---")
    inspector.handle_missing_values(column="Age", strategy="median")
    inspector.handle_missing_values(column="Fare", strategy="mean")
    inspector.handle_missing_values(column="Sex", strategy="mode")
    inspector.handle_missing_values(column="Embarked", strategy="mode")

    # Handle dataset outliers gracefully using IQR filters
    print("\n--- HANDLING OUTLIERS ---")
    inspector.handle_outliers(column="Age", action="delete")

    # Verification Step
    print("\n--- RE-CHECKING CLEANED STRUCTURE ---")
    inspector.display_summary()

    # Step C: Normalization and Encoding Preparations
    print("\n--- STEP C: FEATURE NORMALIZATION & ENCODING ---")
    scaled_num = inspector.extract_normalized_numeric_data(columns=["Age", "Fare"], strategy="standard")
    encoded_cat = inspector.extract_normalized_categorical_data(columns=["Sex", "Embarked"], strategy="onehot")

    print("📊 Scaled Numeric Feature View Preview:")
    display(scaled_num.head())

    print("\n📊 Encoded Categorical Feature View Preview:")
    display(encoded_cat.head())

    # Build final combined analysis modeling workspace
    unified_ml_df = inspector.merge_features(scaled_num, encoded_cat)
    print("\n💼 Combined Modeling Master DataFrame:")
    display(unified_ml_df.head())

    # Step D: Advanced Multi-type Visualizations
    print("\n--- STEP D: VISUALIZATION DEMONSTRATIONS ---")
    # 1. Univariate Plot
    inspector.plot_univariate_subplots(column="Fare")

    # 2. Variable Relationships
    inspector.plot_relationship(col_x="Pclass", col_y="Fare")  # Cat vs Num
    inspector.plot_relationship(col_x="Sex", col_y="Embarked")  # Cat vs Cat

    # 3. Categorical Frequencies
    inspector.plot_categorical_frequency(column="Embarked")

    # 4. Association Matrix Across Diverse Data Formats
    inspector.plot_all_associations_heatmap()

    # Step E: Modular HTML Export Proof of Concept
    print("\n--- STEP E: MODULAR HTML GENERATION PROOF ---")
    summary_counts = inspector.df["Sex"].value_counts().reset_index()
    summary_counts.columns = ["Sex", "Count"]

    html_snippet = PlottingMethods.generate_bar_chart(
        df=summary_counts, x="Sex", y="Count", title="Modular Component Plot"
    )
    print(f"📦 Successfully exported standalone chart to HTML String snippet. Length: {len(html_snippet)} chars.")

🚀 Initializing Real-world Test Using Simulated Titanic Dataset...

--- STEP A: DATA INGESTION & TYPE CORRECTION ---
--------------------------------------------------
📊 STRUCTURAL DATA SUMMARY
--------------------------------------------------
Total Rows:    8
Total Columns: 8

🔢 Numerical Columns (5):
   ['PassengerId', 'Survived', 'Pclass', 'Age', 'Fare']

🔤 Categorical Columns (3):
   ['Name', 'Sex', 'Embarked']

📋 First 20 Rows Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,Thisara,male,22.0,7.25,S
1,2,1,1,Nethmi,female,38.0,71.28,C
2,3,1,3,jessica,female,26.0,7.92,S
3,4,1,1,Sriyani,female,35.0,53.10,S
4,5,0,3,Allen,male,35.0,8.05,S
5,6,0,3,Pasindu,male,NaN,8.46,Q
6,7,0,1,Sampath,male,54.0,51.86,NaN
7,8,1,3,Johnson,NaN,160.0,NaN,S


--------------------------------------------------

--- STEP B: DATA CLEANING & IMPUTATION ---
✨ Imputed missing entries in 'Age' using 'median' strategy (35.0).
✨ Imputed missing entries in 'Fare' using 'mean' strategy (29.702857142857145).
✨ Imputed missing entries in 'Sex' using 'mode' strategy (male).
✨ Imputed missing entries in 'Embarked' using 'mode' strategy (S).

--- HANDLING OUTLIERS ---
🗑️ Dropped 1 rows containing outliers in 'Age'.

--- RE-CHECKING CLEANED STRUCTURE ---
--------------------------------------------------
📊 STRUCTURAL DATA SUMMARY
--------------------------------------------------
Total Rows:    7
Total Columns: 8

🔢 Numerical Columns (5):
   ['PassengerId', 'Survived', 'Pclass', 'Age', 'Fare']

🔤 Categorical Columns (3):
   ['Name', 'Sex', 'Embarked']

📋 First 20 Rows Preview:


,PassengerId,Survived,Pclass,Name,Sex,Age,Fare,Embarked
0,1,0,3,Thisara,male,22.0,7.25,S
1,2,1,1,Nethmi,female,38.0,71.28,C
2,3,1,3,jessica,female,26.0,7.92,S
3,4,1,1,Sriyani,female,35.0,53.10,S
4,5,0,3,Allen,male,35.0,8.05,S
5,6,0,3,Pasindu,male,35.0,8.46,Q
6,7,0,1,Sampath,male,54.0,51.86,S


--------------------------------------------------

--- STEP C: FEATURE NORMALIZATION & ENCODING ---
📊 Scaled Numeric Feature View Preview:


...


💼 Combined Modeling Master DataFrame:


,PassengerId,Survived,Pclass,Age,Fare,Age_standard,Fare_standard,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,22.0,7.25,-1.381327,-0.869681,0,1,0,0,1
1,2,1,1,38.0,71.28,0.318768,1.610433,1,0,1,0,0
2,3,1,3,26.0,7.92,-0.956303,-0.843729,1,0,0,0,1
3,4,1,1,35.0,53.10,0.000000,0.906256,1,0,0,0,1
4,5,0,3,35.0,8.05,0.000000,-0.838694,0,1,0,0,1



--- STEP D: VISUALIZATION DEMONSTRATIONS ---


/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: ConstantInputWarning:

Each of the input arrays is constant; the F statistic is not defined or infinite




--- STEP E: MODULAR HTML GENERATION PROOF ---
📦 Successfully exported standalone chart to HTML String snippet. Length: 8367 chars.
